In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
Example: Running HaMER inference on a sample video in a Jupyter notebook,
with the corrected import statements.
"""

import sys
import os
import cv2
import torch
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML
from IPython.display import display

# 1. Add HaMER to the Python path (adjust if your path is different)
hamer_repo_path = "/home/robotics/Desktop/hamer"  # or wherever you cloned hamer
sys.path.append(hamer_repo_path)
sys.path.append(os.path.join(hamer_repo_path, "hamer"))

# 2. Import relevant HaMER modules
try:
    # Import the convenience function to load the model
    from hamer.models import load_hamer
    # You could also do: from hamer.models.hamer import HAMER (then manually load the checkpoint)
except ImportError as e:
    print("Could not import HaMER modules. Make sure your paths and environment are set up correctly.")
    raise e

# 3. Define the paths
video_input_path = os.path.join(hamer_repo_path, "example_data", "sample_video.mp4")
output_dir = os.path.join(hamer_repo_path, "example_data", "output_hamer")
os.makedirs(output_dir, exist_ok=True)

# The default or custom checkpoint path:
# The __init__.py suggests DEFAULT_CHECKPOINT=f'{CACHE_DIR_HAMER}/hamer_ckpts/checkpoints/hamer.ckpt'
# So if you have a custom path, specify it here:
checkpoint_path = "/home/robotics/Desktop/hamer/_DATA/hamer_ckpts/checkpoints/hamer.ckpt"

# 4. Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# The load_hamer function returns (model, model_cfg)
model, model_cfg = load_hamer(checkpoint_path=checkpoint_path)
model = model.to(device)
model.eval()

print("HaMER model loaded successfully.")

# 5. Open the video
cap = cv2.VideoCapture(video_input_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video Info - {frame_count} frames, {fps} fps, resolution {width}x{height}")

# Prepare an output video writer if you want to save overlays
output_video_path = os.path.join(output_dir, "hamer_output.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

all_3d_outputs = []

frame_index = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert BGR -> RGB if needed for your transforms
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # --------------------------
    # 6. Preprocess the frame for HaMER
    #    Typically, you'll need to do something like:
    #    - Crop/resize to (256,256) if your model_cfg says so
    #    - Convert to torch.Tensor and normalize

    # PSEUDO steps (you must adapt to your config):
    # import torchvision.transforms as transforms
    # transform = transforms.Compose([
    #     transforms.ToPILImage(),
    #     transforms.Resize((256,256)),
    #     transforms.ToTensor(),
    #     transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]) # example
    # ])
    # input_tensor = transform(frame_rgb).unsqueeze(0).to(device)

    # For demonstration, let's skip real transforms:
    input_tensor = torch.zeros(1, 3, 256, 256).to(device)  # dummy data

    with torch.no_grad():
        # Run forward pass. The HaMER model forward method might return a dict with
        # SMPL parameters, 3D joints, etc. e.g.:
        # output = model(input_tensor)
        output = {
            "joints_3d": torch.zeros((1, 24, 3), device=device) + frame_index
        }

    # Extract predicted 3D joints from model output
    predictions_3d = output["joints_3d"][0].cpu().numpy()  # shape (24, 3)
    all_3d_outputs.append(predictions_3d)

    # If you want to overlay skeleton in 2D:
    # frame_vis = visualize_2d_skeleton(frame, predictions_3d) # pseudo function
    frame_vis = frame

    out_writer.write(frame_vis)
    frame_index += 1

cap.release()
out_writer.release()
print(f"Inference complete. Video with 2D overlays saved at: {output_video_path}")

# 7. Simple 3D Visualization with Plotly
final_frame_3d = all_3d_outputs[-1]
x_vals = final_frame_3d[:, 0]
y_vals = final_frame_3d[:, 1]
z_vals = final_frame_3d[:, 2]

fig = go.Figure(data=[go.Scatter3d(
    x=x_vals,
    y=y_vals,
    z=z_vals,
    mode='markers',
    marker=dict(size=4, color='red'),
    name='HaMER_3D_Points'
)])
fig.update_layout(
    title="HaMER 3D Pose (Example)",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)
fig.show()

plot_save_path = os.path.join(output_dir, "hamer_3d_pose.html")
fig.write_html(plot_save_path)
print(f"Saved 3D pose Plotly figure to {plot_save_path}")

print("Done!")
